# Scaled Dot-Product Attention
*The mechanism that lets every token "look at" every other token.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_online_per_query.py` → `sm89_v2_flash_tiled_kv.py`


## What Is Attention?

**In plain English:** Attention lets each word in a sentence consult every other word to understand its meaning in context. The word "bank" in "I sat by the river bank" should look at "river" more than "money".

Each token produces three things:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I advertise about myself?"
- **Value (V):** "What do I actually contribute if selected?"

A query dot-products with all keys to get scores → softmax makes them probabilities → those probabilities weight the values → output is a blend of all values.


In [ ]:
import math
print('ready')

## Worked Example: Three Tokens

We have 3 tokens with $d_\text{head} = 2$ dimensions each.

$$Q = \begin{bmatrix} 0.5 & 0.1 \\ 0.3 & 0.8 \\ 0.2 & 0.6 \end{bmatrix} \quad K = \begin{bmatrix} 0.9 & 0.2 \\ 0.1 & 0.7 \\ 0.8 & 0.3 \end{bmatrix} \quad V = \begin{bmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \\ 0.5 & 0.5 \end{bmatrix}$$

Rows are tokens 0, 1, 2 ("cat", "sat", "mat"). Columns are features.

### Step 1: Compute Score Matrix $S = QK^T / \sqrt{d}$

Scale $= 1/\sqrt{2} \approx 0.707$

$S_{t,s} = \frac{1}{\sqrt{2}} \sum_{k} Q_{t,k} \cdot K_{s,k}$

For token 2 ("mat") attending to all tokens:

| Pair | Dot product | × scale | Score |
|------|-------------|---------|-------|
| mat→cat | $0.2 \times 0.9 + 0.6 \times 0.2 = 0.30$ | × 0.707 | 0.212 |
| mat→sat | $0.2 \times 0.1 + 0.6 \times 0.7 = 0.44$ | × 0.707 | 0.311 |
| mat→mat | $0.2 \times 0.8 + 0.6 \times 0.3 = 0.34$ | × 0.707 | 0.240 |


In [ ]:
Q = [[0.5, 0.1], [0.3, 0.8], [0.2, 0.6]]
K = [[0.9, 0.2], [0.1, 0.7], [0.8, 0.3]]
V = [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]]
T, d = 3, 2
scale = 1.0 / math.sqrt(d)

# Compute full score matrix
S = [[scale * sum(Q[t][k] * K[s][k] for k in range(d))
      for s in range(T)] for t in range(T)]

print("Score matrix S = Q @ K.T / sqrt(d):")
print("         cat    sat    mat")
for t, row in enumerate(S):
    label = ["cat","sat","mat"][t]
    print(f"  {label}:  {row[0]:.3f}  {row[1]:.3f}  {row[2]:.3f}")


### Step 2: Causal Mask (Decoder / Autoregressive)

A decoder model must not let token $t$ peek at future tokens $s > t$.

$$S_{t,s} \leftarrow \begin{cases} S_{t,s} & s \le t \\ -\infty & s > t \end{cases}$$

Since $e^{-\infty} = 0$, future tokens get zero probability after softmax.

$$\text{Masked score matrix:} \quad \begin{bmatrix} 0.403 & -\infty & -\infty \\ 0.191 & 0.389 & -\infty \\ 0.212 & 0.311 & 0.240 \end{bmatrix}$$

### 🗣️ Why Divide by $\sqrt{d}$?

> *"If $q$ and $k$ are random unit-variance vectors of length $d$, their dot product has variance $d$ — so larger $d$ pushes scores into regions where softmax is nearly flat (near-uniform). Dividing by $\sqrt{d}$ restores variance to 1, keeping gradients healthy."*

### Step 3: Softmax Row-by-Row → Attention Weights $P$

For token 2 (mat): scores $= [0.212, 0.311, 0.240]$

$$P_{2,:} = \text{softmax}([0.212, 0.311, 0.240]) = [0.319, 0.353, 0.328]$$

### Step 4: Output = $P \cdot V$

$$y_\text{mat} = 0.319 \cdot [1,0] + 0.353 \cdot [0,1] + 0.328 \cdot [0.5, 0.5]$$
$$= [0.319, 0] + [0, 0.353] + [0.164, 0.164] = [0.483, 0.517]$$


In [ ]:
def softmax(row):
    m = max(v for v in row if v != float('-inf'))
    exps = [math.exp(v - m) if v != float('-inf') else 0.0 for v in row]
    s = sum(exps)
    return [e/s for e in exps]

# Apply causal mask
S_masked = [[S[t][s] if s <= t else float('-inf')
             for s in range(T)] for t in range(T)]

print("Masked scores:")
labels = ["cat","sat","mat"]
for t, row in enumerate(S_masked):
    vals = [f"{v:.3f}" if v != float('-inf') else " -inf" for v in row]
    print(f"  {labels[t]}: {vals}")

# Softmax to get attention weights P
P = [softmax(row) for row in S_masked]
print("\nAttention weights P:")
for t, row in enumerate(P):
    print(f"  {labels[t]}: {[round(v,3) for v in row]}")

# Output = P @ V
Y = [[sum(P[t][s] * V[s][d] for s in range(T)) for d in range(2)] for t in range(T)]
print("\nOutput Y:")
for t, row in enumerate(Y):
    print(f"  {labels[t]}: {[round(v,3) for v in row]}")
print("\n✓ At t=0 (cat, causal), output = V[0] exactly:", [round(v,6) for v in Y[0]])
assert all(abs(Y[0][d] - V[0][d]) < 1e-9 for d in range(2))


## The Formula

$$\boxed{\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V}$$

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $Q$ | $(T, d_k)$ | Query matrix — what each token seeks |
| $K$ | $(T, d_k)$ | Key matrix — what each token offers |
| $V$ | $(T, d_v)$ | Value matrix — what each token contributes |
| $QK^T$ | $(T, T)$ | All pairwise query-key dot products |
| $d_k$ | scalar | Head dimension; dividing by $\sqrt{d_k}$ stabilizes gradients |
| $\text{softmax}(\cdots)$ | $(T, T)$ | Attention weights; each row sums to 1 |

### 🗣️ Say It Out Loud
> *"Attention of Q, K, V equals: softmax of Q times K-transpose divided by square root of d-sub-k, all times V."*

**Tutor's gloss:** "$QK^T$ gives us a score matrix: row $t$, column $s$ is 'how similar is query $t$ to key $s$?' Dividing by $\sqrt{d_k}$ stops the scores from getting too large. Softmax turns scores into probabilities. Multiplying by $V$ blends the value vectors using those probabilities."


## FlashAttention: Avoiding the $T \times T$ Matrix

**The memory problem:** $QK^T \in \mathbb{R}^{T \times T}$ at $T=4096$ is $64\,\text{MB}$ per head (fp32). Written to HBM, read back twice. For long sequences: infeasible.

**FlashAttention insight (Dao et al., 2022):** Use online softmax to compute attention without materializing $QK^T$:

Process $K$ and $V$ in tiles of size $B_c$:

```
for each KV tile (k_start, k_start + Bc):
    load K[k_start:k_start+Bc, :] into SMEM   ← read ONCE
    load V[k_start:k_start+Bc, :] into SMEM   ← read ONCE
    for each position k_local in tile:
        score = Q[t] · K[k_local] / sqrt(d)
        online_update(m, d, acc, score, V[k_local])
output[t] = acc / d
```

**HBM read comparison:**

| Version | Reads K+V | Memory for scores | Algorithm |
|---------|-----------|-------------------|-----------|
| `v0_naive` | $O(T^2 D)$ | $O(T^2)$ written to HBM | 2-pass, re-reads K+V |
| `sm89_v2_flash_tiled_kv` | $O(T D)$ | $O(1)$ in registers | Online, tiles in SMEM |

At $T=4096$: **4096× less K/V data read.**
